# Session 2: Exploratory Data Analysis (Part A) — Retail Transactions Inspection

## What We Will Cover
This notebook focuses on the initial data ingestion, structural validation, and quality inspection of the core daily transaction logs. Before building any predictive supply chain models, we must understand our data's foundational shape, lifecycle patterns, and structural integrity.

### 1. Ingestion & Shape Verification
* **Data Dimension Auditing:** Loading `transactions.csv` and evaluating row-to-column dimensions to establish baseline data expectations.
* **Granularity Assessment:** Identifying what an individual record represents (daily product sales metrics).

### 2. Feature Schema & Typology Audits
* **Data Type Discovery:** Identifying mismatched formats (e.g., string text representing dates or categorical attributes) that could trigger downstream mathematical errors.
* **Data Quality Verification:** Scanning for data missingness (Null/NaN values) and row duplication to protect model inputs from artificial distortion.

### 3. Structural Variance & Redundancy Cleaning
* **Global vs. Conditional Variance:** Isolating features that look dynamic globally but are strictly static *per product* (e.g., product names, retail categories, unit costs, and unit prices).
* **Schema Optimization:** Systematically dropping redundant columns whose variance is 100% explained by relational keys (`product_id`), streamlining the dataset into a lean, highly efficient transaction table.

### 4. Cross-Column Logic Validation
* **Mathematical Sanity Checks:** Writing programmatic integrity rules to verify that financial equations hold perfectly consistent across every single entry ($Gross\ Profit = Revenue - COGS$).

In [11]:
# Import the essential libraries
import pandas as pd
import numpy as np

In [12]:
# Read from dataset
transactions_data = pd.read_csv('../../data/transactions.csv')
products_data = pd.read_csv('../../data/products.csv')

In [13]:
# Displaying the first five rows from dataset
transactions_data.head()

,date,product_id,product_name,category,units_sold,unit_cost,unit_price,revenue,cogs,gross_profit
0,2024-01-01,P001,Wireless Headphones,Electronics,15,45.0,89.99,1349.85,675.0,674.85
1,2024-01-01,P002,Yoga Mat,Fitness,16,12.0,29.99,479.84,192.0,287.84
2,2024-01-01,P003,Stainless Water Bottle,Kitchen,17,8.0,19.99,339.83,136.0,203.83
3,2024-01-01,P004,Bluetooth Speaker,Electronics,12,30.0,59.99,719.88,360.0,359.88
4,2024-01-01,P005,Winter Jacket,Apparel,5,55.0,129.99,649.95,275.0,374.95


In [14]:
# Displaying the first five rows from the dataset
products_data.head()

,product_id,name,category,unit_cost,unit_price,base_demand
0,P001,Wireless Headphones,Electronics,45.0,89.99,22
1,P002,Yoga Mat,Fitness,12.0,29.99,18
2,P003,Stainless Water Bottle,Kitchen,8.0,19.99,30
3,P004,Bluetooth Speaker,Electronics,30.0,59.99,15
4,P005,Winter Jacket,Apparel,55.0,129.99,10


In [15]:
# Shape of dataset
transactions_data.shape

(1825, 10)

Each row in the transactions dataset represents daily sales performance for a specific product, tracking units sold, pricing, revenue, cost, and profit.

In [16]:
# List of columns
transactions_data.columns.tolist()

['date',
 'product_id',
 'product_name',
 'category',
 'units_sold',
 'unit_cost',
 'unit_price',
 'revenue',
 'cogs',
 'gross_profit']

### Each column meaning
1. 'data': The specific date when the transaction occurred.
2. 'product_id': Unique identifier for each product.
3. 'product_name': The commerical name of the item sold.
4. 'category': The department where each product belogns to.
5. 'units_sold': Quantity of that item sold on that day.
6. 'unit_cost': The amount it costs your buisness to acquire or manufacture a single unit of the product from the supplier.
7. 'unit_price': The retail price at which you sell a single unit.
8. 'revenue': The total gross income generated from sales of this product on that day.
9. 'cogs': The total direct cost to business for that inventory that was sold.
10. 'gross_profit': The amount of money left over from revenue after deducting the direct cost of the goods.   

In [17]:
# Dtypes of dataset
transactions_data.dtypes

date             object
product_id       object
product_name     object
category         object
units_sold        int64
unit_cost       float64
unit_price      float64
revenue         float64
cogs            float64
gross_profit    float64
dtype: object

### Columns Requiring Data Type Conversion
1. data (from object to date)
2. category (from object to category)

In [18]:
# Finding out if there are any null values
transactions_data.isnull().sum()

date            0
product_id      0
product_name    0
category        0
units_sold      0
unit_cost       0
unit_price      0
revenue         0
cogs            0
gross_profit    0
dtype: int64

In [19]:
# Finding out if there are any duplicate values
transactions_data.duplicated().sum()

np.int64(0)

A completely clean result with zero nuk=ll and zero duplicates tells a few important things about the dataset:
1. **High Data completeness and quality:** 
2. **Data integrity**
3. **Ready for exploration**

In [20]:
# Date range of transactions_data and no of transactions per day
transactions_data['date'] = pd.to_datetime(transactions_data['date'], format='mixed')

print(transactions_data['date'].min())
print(transactions_data['date'].max())

transactions_per_date = transactions_data.shape[0] / transactions_data['date'].nunique()
print(f"No of transactions per date: {int(transactions_per_date)}.")

2024-01-01 00:00:00
2024-12-30 00:00:00
No of transactions per date: 5.


A full year of data (365 days) featuring exactly 5 product sales records per day.

In [21]:
# No of unique values 
transactions_data['product_id'].nunique()

5

In [22]:
# Total Occurence of each product_id 
transactions_data['product_id'].value_counts()

product_id
P001    365
P002    365
P003    365
P004    365
P005    365
Name: count, dtype: int64

The distribution is perfectly balanced. Each of the 5 unique products appears exactly 365 times, representing one sales record per day for the entire year.

In [23]:
# no of unique values
transactions_data['product_name'].nunique()

5

In [24]:
# unique values of product_name split up by product_id
transactions_data.groupby('product_id')['product_name'].nunique().reset_index()

,product_id,product_name
0,P001,1
1,P002,1
2,P003,1
3,P004,1
4,P005,1


In [25]:
# droping the column
transactions_data = transactions_data.drop(columns=['product_name'])

In [26]:
# no of unique values
transactions_data['category'].nunique()

4

In [27]:
# unique values of category split up by product_id
transactions_data.groupby('product_id')['category'].nunique().reset_index()

,product_id,category
0,P001,1
1,P002,1
2,P003,1
3,P004,1
4,P005,1


In [28]:
# dropping the column
transactions_data = transactions_data.drop(columns=['category'])

In [29]:
# Displaying the first five rows from dataset
transactions_data.head()

,date,product_id,units_sold,unit_cost,unit_price,revenue,cogs,gross_profit
0,2024-01-01,P001,15,45.0,89.99,1349.85,675.0,674.85
1,2024-01-01,P002,16,12.0,29.99,479.84,192.0,287.84
2,2024-01-01,P003,17,8.0,19.99,339.83,136.0,203.83
3,2024-01-01,P004,12,30.0,59.99,719.88,360.0,359.88
4,2024-01-01,P005,5,55.0,129.99,649.95,275.0,374.95


In [30]:
# no of unique values
transactions_data[['unit_cost', 'unit_price']].nunique()

unit_cost     5
unit_price    5
dtype: int64

In [31]:
# unique values of unit_cost and unit_price split up by product_id
transactions_data.groupby('product_id')[['unit_cost', 'unit_price']].nunique().reset_index()

,product_id,unit_cost,unit_price
0,P001,1,1
1,P002,1,1
2,P003,1,1
3,P004,1,1
4,P005,1,1


In [32]:
# dropping the columns
transactions_data = transactions_data.drop(columns=['unit_cost', 'unit_price'])

In [33]:
# summary of units_sold column
transactions_data['units_sold'].describe().reset_index()

,index,units_sold
0,count,1825.000000
1,mean,22.332603
2,std,10.818861
3,min,4.000000
4,25%,14.000000
5,50%,21.000000
6,75%,29.000000
7,max,72.000000


* **Min:** 4 | **Max:** 72 | **Mean:** 22.33 | **Std Dev:** 10.82

**Conclusion:** Demand is **bumpy**. The high standard deviation relative to the mean and the wide gap between the minimum (4) and maximum (72) sales indicate significant day-to-day fluctuations in customer demand.

In [34]:
# verifying the gross_profit column
is_consistent = np.isclose(transactions_data['gross_profit'], transactions_data['revenue'] - transactions_data['cogs'])
print(f"Are all rows mathematically consistent? {is_consistent.all()}")

Are all rows mathematically consistent? True


In [35]:
# A whole dataset
products_data

,product_id,name,category,unit_cost,unit_price,base_demand
0,P001,Wireless Headphones,Electronics,45.0,89.99,22
1,P002,Yoga Mat,Fitness,12.0,29.99,18
2,P003,Stainless Water Bottle,Kitchen,8.0,19.99,30
3,P004,Bluetooth Speaker,Electronics,30.0,59.99,15
4,P005,Winter Jacket,Apparel,55.0,129.99,10


In [36]:
# no of rows from the dataset
products_data.shape[0]

5

* **Total Rows:** 5 (one unique row for each `product_id` from P001 to P005).

**File Role:**
This file acts as a **reference / lookup table** (also known as a *Dimension table* in database design). 

* **Why it's a reference table:** It contains static, unchanging attributes for each item—such as the official product name, category, standard unit cost, retail price, and base demand. 
* **Why it's NOT a time-series table:** Unlike `transactions.csv`, it does not contain a date column and does not track daily changes or individual event histories. It serves as the master directory to look up details whenever a `product_id` appears in other datasets.

In [37]:
# values of category column
products_data['category']

0    Electronics
1        Fitness
2        Kitchen
3    Electronics
4        Apparel
Name: category, dtype: object

In [38]:
# no of unique values
products_data.groupby('product_id')['category'].nunique().reset_index()

,product_id,category
0,P001,1
1,P002,1
2,P003,1
3,P004,1
4,P005,1


* **The Four Categories:** Electronics, Fitness, Kitchen, and Apparel.

**Category Sharing Evaluation:**
Yes, there is a shared category in the dataset. While Fitness, Kitchen, and Apparel each contain only a single unique product, the **Electronics** category is shared between two distinct products:
1. `P001` (Wireless Headphones)
2. `P004` (Bluetooth Speaker)

Your groupby analysis confirms that while categories can be shared among different items, each individual product ID maps perfectly to exactly one unique category.

In [39]:
# figuring out the profit margin
products_data['margin_pct'] = ((products_data['unit_price'] - products_data['unit_cost']) * 100) / products_data['unit_price']
products_data[['product_id', 'name', 'margin_pct']]
products_data[['name', 'margin_pct']].sort_values(by='margin_pct', ascending=False)

,name,margin_pct
1,Yoga Mat,59.986662
2,Stainless Water Bottle,59.979990
4,Winter Jacket,57.689053
0,Wireless Headphones,49.994444
3,Bluetooth Speaker,49.991665


**Ranking from Highest to Lowest Margin:**
1. **Yoga Mat:** **59.99%** (Highest)
2. **Stainless Water Bottle:** **59.98%**
3. **Winter Jacket:** **57.69%**
4. **Wireless Headphones:** **49.99%**
5. **Bluetooth Speaker:** **49.99%** (Lowest)

### Key Takeaways:
* **Highest Margin:** The **Yoga Mat** yields the highest return per dollar of revenue at **59.99%**, closely followed by the Stainless Water Bottle.
* **Lowest Margin:** The **Bluetooth Speaker** has the lowest profit margin at **49.99%** (trailing the Wireless Headphones by a fraction of a decimal due to floating-point precision). 
* **Strategic Note:** Interestingly, the items with the highest retail prices (Winter Jacket and Wireless Headphones) do not have the highest profit margins. The lower-cost utility items (Yoga Mat and Water Bottle) are actually more efficient profit-generators per sale.

* **Definition:** `base_demand` represents the expected average daily sales volume for a product under normal conditions, acting as the anchor baseline before random daily fluctuations are applied.
* **Range of Values:** 10 to 30 units/day.

**Product Breakdown:**
* **Highest Base Demand:** `P003` (Stainless Water Bottle) at **30 units/day**.
* **Lowest Base Demand:** `P005` (Winter Jacket) at **10 units/day**.

### Inventory Risk Implications:
A **high base demand** (like the Stainless Water Bottle) alters inventory risk in two ways:
1. **Decreased Obsolescence Risk:** The product moves fast, meaning inventory turns over quickly and is highly unlikely to become dead stock.
2. **Increased Stockout Risk:** Because the velocity of sales is so fast, any delay from suppliers or unexpected demand spikes leaves a very small margin for error, significantly increasing the risk of running out of stock if safety levels aren't carefully managed.

In [40]:
# dropping the columns with same value
zero_variance_mask = transactions_data.nunique() == 1
cols_to_drop = zero_variance_mask[zero_variance_mask].index

if len(cols_to_drop) > 0:
    transactions_data.drop(columns=cols_to_drop, inplace=True)
    print(f"Dropped zero-variance columns: {list(cols_to_drop)}")
else:
    print("No zero variance columns found")

No zero variance columns found


**1. Columns with exactly 5 unique values:**
Before the initial data cleanup, the columns that contained exactly **5 unique values** (matching the total number of unique products) were:
* `product_name`
* `unit_cost`
* `unit_price`
*(Note: `category` contained 4 unique values because two products shared the "Electronics" category).*

**2. Does having 5 unique values make them zero-variance?**
**No, not globally.** Your test in cell [104] correctly returned `"No zero variance columns found"` because a true **zero-variance** column requires *every single row in the entire file* to contain the exact same value (e.g., if every transaction across all dates showed a cost of $45.00). Because these columns contain 5 distinct values across the 1,825 rows, global mathematical variance is present.

**3. The Verdict on Redundancy:**
While they are not globally zero-variance, they are **conditionally zero-variance per product**. This means once you group by a specific `product_id`, the values never change. 

Because their variance is 100% explained by the `product_id` column, they provide absolutely no new behavioral information or predictive signals to a machine learning model or statistical analysis. Dropping them from the daily transactions table was the mathematically correct step to eliminate data redundancy.

In [41]:
# list of finalized columns columns 
transactions_data.columns.tolist()

['date', 'product_id', 'units_sold', 'revenue', 'cogs', 'gross_profit']

**Columns Dropped:**
* **`product_name`** (Dropped in Cell [81]): Redundant text string; every `product_id` maps perfectly to exactly one product name in the reference table.
* **`category`** (Dropped in Cell [84]): Redundant department attribute; it is a static property of the item itself rather than a daily transaction variable.
* **`unit_cost` & `unit_price`** (Dropped in Cell [88]): Removed because both cost and retail price remained completely fixed and unchanging for each product throughout the entire year.

### Final Clean Transactions Schema:
As verified by the column list in Cell [106], our finalized transactional dataframe contains:
1. `date` — The transaction timestamp.
2. `product_id` — The relational key to look up product details.
3. `units_sold` — The daily sales volume metric.
4. `revenue` — Total daily gross income.
5. `cogs` — Total daily cost of goods sold.
6. `gross_profit` — Net daily profit performance.

In [43]:
# updating the csv file
transactions_data.to_csv('../../data/transactions.csv', index=False)